# Notebook 17 – Mini Assessment


## Question 1 – What is Feature Engineering?

Explain Feature Engineering and why it is important in Machine Learning.

In [1]:
# Example: Create a new feature from an existing feature

import pandas as pd

df = pd.DataFrame({
    "tenure": [12, 24, 36, 48, 60]
})

df["TenureYears"] = df["tenure"] / 12

display(df)

,tenure,TenureYears
0,12,1.0
1,24,2.0
2,36,3.0
3,48,4.0
4,60,5.0


## Question 2 – Feature Engineering vs Feature Selection

Explain the difference between Feature Engineering and Feature Selection.

In [2]:
from sklearn.feature_selection import SelectKBest, f_classif

data = pd.DataFrame({
    "Age": [22, 25, 30, 35, 40, 45],
    "Salary": [20000, 25000, 30000, 35000, 40000, 45000],
    "Experience": [1, 2, 3, 5, 7, 9]
})

# Feature Engineering
data["SalaryPerYear"] = data["Salary"] / (data["Experience"] + 1)

# Feature Selection
X = data
y = [0, 0, 0, 1, 1, 1]

selector = SelectKBest(score_func=f_classif, k=2)
selected_data = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Engineered Dataset:")
display(data)

print("Selected Features:")
print(selected_features.tolist())

Engineered Dataset:


,Age,Salary,Experience,SalaryPerYear
0,22,20000,1,10000.000000
1,25,25000,2,8333.333333
2,30,30000,3,7500.000000
3,35,35000,5,5833.333333
4,40,40000,7,5000.000000
5,45,45000,9,4500.000000


Selected Features:
['Experience', 'SalaryPerYear']


## Question 3 – Why is Domain Knowledge Important?

Explain why domain knowledge is important in Feature Engineering.

In [3]:
customer_df = pd.DataFrame({
    "TotalPurchases": [10, 20, 5, 15, 30],
    "TotalAmount": [5000, 10000, 2000, 7500, 15000],
    "CustomerAge": [25, 30, 22, 35, 40]
})

customer_df["AverageOrderValue"] = (
    customer_df["TotalAmount"] / customer_df["TotalPurchases"]
)

customer_df["PurchaseCategory"] = pd.cut(
    customer_df["TotalPurchases"],
    bins=[0, 10, 20, float("inf")],
    labels=["Low", "Medium", "High"]
)

display(customer_df)

,TotalPurchases,TotalAmount,CustomerAge,AverageOrderValue,PurchaseCategory
0,10,5000,25,500.0,Low
1,20,10000,30,500.0,Medium
2,5,2000,22,400.0,Low
3,15,7500,35,500.0,Medium
4,30,15000,40,500.0,High


## Question 4 – What is Feature Leakage?

Explain Feature Leakage and identify possible leakage columns in a dataset.

In [4]:
churn_df = pd.DataFrame({
    "tenure": [12, 24, 36, 48],
    "MonthlyCharges": [50, 80, 100, 120],
    "Churn": [0, 1, 0, 1],
    "CancellationDate": [
        None, "2026-01-10", None, "2026-02-15"
    ]
})

target_column = "Churn"

suspicious_columns = [
    column for column in churn_df.columns
    if "cancellation" in column.lower()
]

X = churn_df.drop(
    columns=[target_column] + suspicious_columns
)

y = churn_df[target_column]

print("Input Features:")
display(X)

print("Target:")
display(y)

print("Suspicious Leakage Columns:", suspicious_columns)

Input Features:


,tenure,MonthlyCharges
0,12,50
1,24,80
2,36,100
3,48,120


Target:


0    0
1    1
2    0
3    1
Name: Churn, dtype: int64

Suspicious Leakage Columns: ['CancellationDate']


## Question 5 – What is an Interaction Feature?

Create interaction features using Tenure and MonthlyCharges.

In [5]:
interaction_df = pd.DataFrame({
    "Tenure": [12, 24, 36, 48],
    "MonthlyCharges": [50, 80, 100, 120]
})

interaction_df["TenureMonthlyInteraction"] = (
    interaction_df["Tenure"]
    * interaction_df["MonthlyCharges"]
)

display(interaction_df)

,Tenure,MonthlyCharges,TenureMonthlyInteraction
0,12,50,600
1,24,80,1920
2,36,100,3600
3,48,120,5760


## Question 6 – What is Binning?

Create age groups using numerical age values.

In [6]:
age_df = pd.DataFrame({
    "Age": [18, 22, 27, 35, 42, 50, 61, 70]
})

age_df["AgeGroup"] = pd.cut(
    age_df["Age"],
    bins=[17, 25, 40, 60, float("inf")],
    labels=["Young", "Adult", "Middle-Aged", "Senior"]
)

display(age_df)

,Age,AgeGroup
0,18,Young
1,22,Young
2,27,Adult
3,35,Adult
4,42,Middle-Aged
5,50,Middle-Aged
6,61,Senior
7,70,Senior


## Question 7 – What is Feature Cardinality?

Calculate the number of unique values in each categorical feature.

In [7]:
cardinality_df = pd.DataFrame({
    "City": ["Chennai", "Mumbai", "Delhi", "Pune", "Chennai"],
    "Category": ["Electronics", "Clothing", "Books", "Books", "Clothing"],
    "PaymentMethod": ["UPI", "Card", "Cash", "UPI", "Card"]
})

for column in cardinality_df.columns:
    print(column, ":", cardinality_df[column].nunique())

City : 4
Category : 3
PaymentMethod : 3


## Question 8 – Explain Target Encoding.

Perform Target Encoding on a categorical feature using training data only.

In [8]:
from sklearn.model_selection import train_test_split

target_df = pd.DataFrame({
    "City": [
        "Chennai", "Mumbai", "Delhi", "Chennai",
        "Mumbai", "Delhi", "Chennai", "Mumbai",
        "Delhi", "Chennai", "Mumbai", "Delhi"
    ],
    "Churn": [0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1]
})

train_df, test_df = train_test_split(
    target_df,
    test_size=0.25,
    random_state=42,
    stratify=target_df["Churn"]
)

city_mapping = train_df.groupby("City")["Churn"].mean()
global_mean = train_df["Churn"].mean()

train_df = train_df.copy()
test_df = test_df.copy()

train_df["CityEncoded"] = (
    train_df["City"].map(city_mapping).fillna(global_mean)
)

test_df["CityEncoded"] = (
    test_df["City"].map(city_mapping).fillna(global_mean)
)

print("Training Data:")
display(train_df)

print("Testing Data:")
display(test_df)

Training Data:


,City,Churn,CityEncoded
0,Chennai,0,0.500000
2,Delhi,0,0.333333
11,Delhi,1,0.333333
1,Mumbai,1,0.750000
5,Delhi,0,0.333333
7,Mumbai,1,0.750000
4,Mumbai,1,0.750000
10,Mumbai,0,0.750000
3,Chennai,1,0.500000


Testing Data:


,City,Churn,CityEncoded
8,Delhi,1,0.333333
6,Chennai,0,0.500000
9,Chennai,0,0.500000


## Question 9 – Explain Frequency Encoding.

Perform Frequency Encoding on a categorical feature.

In [9]:
frequency_df = pd.DataFrame({
    "ProductCategory": [
        "Electronics", "Clothing", "Electronics",
        "Books", "Clothing", "Electronics"
    ]
})

frequency_mapping = frequency_df[
    "ProductCategory"
].value_counts()

frequency_df["FrequencyEncoded"] = (
    frequency_df["ProductCategory"].map(frequency_mapping)
)

display(frequency_df)

,ProductCategory,FrequencyEncoded
0,Electronics,3
1,Clothing,2
2,Electronics,3
3,Books,1
4,Clothing,2
5,Electronics,3


## Question 10 – When Should Log Transformation Be Used?

Apply log transformation to a numerical feature and compare the original and transformed values.

In [10]:
log_df = pd.DataFrame({
    "PurchaseAmount": [100, 200, 500, 1000, 5000, 10000, 50000]
})

log_df["LogTransformed"] = np.log1p(
    log_df["PurchaseAmount"]
)

print("Original Data:")
display(log_df[["PurchaseAmount"]])

print("Transformed Data:")
display(log_df)

print("Original Skewness:", log_df["PurchaseAmount"].skew())
print("Transformed Skewness:", log_df["LogTransformed"].skew())

NameError: name 'np' is not defined

## Question 11 – What is the Curse of Dimensionality?

Create a dataset with multiple features and compare its dimensions before and after dimensionality reduction.

In [11]:
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_data, y_data = make_classification(
    n_samples=200,
    n_features=10,
    n_informative=5,
    random_state=42
)

print("Original Shape:", X_data.shape)

scaled_data = StandardScaler().fit_transform(X_data)

pca = PCA(n_components=2)
reduced_data = pca.fit_transform(scaled_data)

print("Reduced Shape:", reduced_data.shape)

Original Shape: (200, 10)
Reduced Shape: (200, 2)


## Question 12 – Explain PCA.

Apply Principal Component Analysis and display the explained variance ratio.

In [12]:
from sklearn.datasets import load_iris

iris = load_iris()

iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

scaled_iris = StandardScaler().fit_transform(iris_df)

pca = PCA(n_components=2)

pca_result = pca.fit_transform(scaled_iris)

pca_df = pd.DataFrame(
    pca_result,
    columns=["PC1", "PC2"]
)

print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("Total Explained Variance:")
print(pca.explained_variance_ratio_.sum())

display(pca_df.head())

Explained Variance Ratio:
[0.72962445 0.22850762]
Total Explained Variance:
0.9581320720000166


,PC1,PC2
0,-2.264703,0.480027
1,-2.080961,-0.674134
2,-2.364229,-0.341908
3,-2.299384,-0.597395
4,-2.389842,0.646835


## Question 13 – What is Feature Importance?

Train a Random Forest model and identify important features.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X_data, y_data = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=4,
    random_state=42
)

feature_names = [f"Feature_{i}" for i in range(1, 9)]

feature_df = pd.DataFrame(
    X_data,
    columns=feature_names
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(feature_df, y_data)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

display(importance_df)

,Feature,Importance
7,Feature_8,0.172279
4,Feature_5,0.159824
6,Feature_7,0.155427
5,Feature_6,0.152287
1,Feature_2,0.135476
0,Feature_1,0.117757
3,Feature_4,0.053518
2,Feature_3,0.053432


## Question 14 – Create Five Useful Features for an E-commerce Dataset

Create five features:

1. TotalAmount
2. DiscountAmount
3. FinalAmount
4. OrderYear
5. OrderMonth

In [14]:
ecommerce_df = pd.DataFrame({
    "Quantity": [2, 5, 3, 4, 1],
    "UnitPrice": [500, 200, 800, 300, 1000],
    "Discount": [10, 5, 15, 0, 20],
    "OrderDate": [
        "2026-01-10", "2026-02-15", "2026-03-20",
        "2026-04-05", "2026-05-12"
    ]
})

ecommerce_df["OrderDate"] = pd.to_datetime(
    ecommerce_df["OrderDate"]
)

ecommerce_df["TotalAmount"] = (
    ecommerce_df["Quantity"] * ecommerce_df["UnitPrice"]
)

ecommerce_df["DiscountAmount"] = (
    ecommerce_df["TotalAmount"] * ecommerce_df["Discount"] / 100
)

ecommerce_df["FinalAmount"] = (
    ecommerce_df["TotalAmount"] - ecommerce_df["DiscountAmount"]
)

ecommerce_df["OrderYear"] = ecommerce_df["OrderDate"].dt.year
ecommerce_df["OrderMonth"] = ecommerce_df["OrderDate"].dt.month

display(ecommerce_df)

,Quantity,UnitPrice,Discount,OrderDate,TotalAmount,DiscountAmount,FinalAmount,OrderYear,OrderMonth
0,2,500,10,2026-01-10,1000,100.0,900.0,2026,1
1,5,200,5,2026-02-15,1000,50.0,950.0,2026,2
2,3,800,15,2026-03-20,2400,360.0,2040.0,2026,3
3,4,300,0,2026-04-05,1200,0.0,1200.0,2026,4
4,1,1000,20,2026-05-12,1000,200.0,800.0,2026,5


## Question 15 – Create Customer-Level Aggregation Features

Calculate for each customer:

1. Total purchase amount.
2. Average purchase amount.
3. Number of orders.
4. Maximum purchase amount.
5. Minimum purchase amount.

In [15]:
transactions = pd.DataFrame({
    "CustomerID": [101, 102, 101, 103, 102, 101, 103],
    "OrderAmount": [500, 800, 300, 1000, 600, 700, 400]
})

customer_features = transactions.groupby(
    "CustomerID"
).agg(
    TotalPurchase=("OrderAmount", "sum"),
    AveragePurchase=("OrderAmount", "mean"),
    OrderCount=("OrderAmount", "count"),
    MaximumPurchase=("OrderAmount", "max"),
    MinimumPurchase=("OrderAmount", "min")
).reset_index()

display(customer_features)

,CustomerID,TotalPurchase,AveragePurchase,OrderCount,MaximumPurchase,MinimumPurchase
0,101,1500,500.0,3,700,300
1,102,1400,700.0,2,800,600
2,103,1400,700.0,2,1000,400


## Question 16 – Extract Features from a Date Column

Extract Year, Month, Day, DayOfWeek, Quarter, and Weekend Indicator from a date column.

In [16]:
date_df = pd.DataFrame({
    "OrderDate": [
        "2026-01-10", "2026-02-15", "2026-03-20",
        "2026-04-05", "2026-05-12"
    ]
})

date_df["OrderDate"] = pd.to_datetime(date_df["OrderDate"])

date_df["Year"] = date_df["OrderDate"].dt.year
date_df["Month"] = date_df["OrderDate"].dt.month
date_df["Day"] = date_df["OrderDate"].dt.day
date_df["DayOfWeek"] = date_df["OrderDate"].dt.day_name()
date_df["Quarter"] = date_df["OrderDate"].dt.quarter
date_df["IsWeekend"] = (
    date_df["OrderDate"].dt.dayofweek >= 5
).astype(int)

display(date_df)

,OrderDate,Year,Month,Day,DayOfWeek,Quarter,IsWeekend
0,2026-01-10,2026,1,10,Saturday,1,1
1,2026-02-15,2026,2,15,Sunday,1,1
2,2026-03-20,2026,3,20,Friday,1,0
3,2026-04-05,2026,4,5,Sunday,2,1
4,2026-05-12,2026,5,12,Tuesday,2,0


## Question 17 – Create Interaction Features

Create interaction features using:

1. Tenure × MonthlyCharges
2. Tenure × TotalCharges
3. MonthlyCharges × TotalCharges

In [17]:
interaction_df = pd.DataFrame({
    "Tenure": [12, 24, 36, 48],
    "MonthlyCharges": [50, 80, 100, 120],
    "TotalCharges": [600, 1920, 3600, 5760]
})

interaction_df["TenureMonthlyInteraction"] = (
    interaction_df["Tenure"] * interaction_df["MonthlyCharges"]
)

interaction_df["TenureTotalInteraction"] = (
    interaction_df["Tenure"] * interaction_df["TotalCharges"]
)

interaction_df["MonthlyTotalInteraction"] = (
    interaction_df["MonthlyCharges"] * interaction_df["TotalCharges"]
)

display(interaction_df)

,Tenure,MonthlyCharges,TotalCharges,TenureMonthlyInteraction,TenureTotalInteraction,MonthlyTotalInteraction
0,12,50,600,600,7200,30000
1,24,80,1920,1920,46080,153600
2,36,100,3600,3600,129600,360000
3,48,120,5760,5760,276480,691200


## Question 18 – Perform Feature Selection Using Multiple Techniques

Perform feature selection using:

1. Variance Threshold
2. SelectKBest
3. Random Forest Feature Importance

Compare the selected features.

In [18]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif

X_data, y_data = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)

feature_names = [f"Feature_{i}" for i in range(1, 11)]

X_df = pd.DataFrame(X_data, columns=feature_names)

# Variance Threshold
variance_selector = VarianceThreshold(threshold=0.01)
variance_selector.fit_transform(X_df)

print("Variance Threshold:")
print(X_df.columns[variance_selector.get_support()].tolist())

# SelectKBest
kbest_selector = SelectKBest(
    score_func=mutual_info_classif,
    k=5
)

kbest_selector.fit_transform(X_df, y_data)

print("\nSelectKBest:")
print(X_df.columns[kbest_selector.get_support()].tolist())

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_df, y_data)

importance_df = pd.DataFrame({
    "Feature": X_df.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nRandom Forest Important Features:")
display(importance_df)

Variance Threshold:
['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5', 'Feature_6', 'Feature_7', 'Feature_8', 'Feature_9', 'Feature_10']

SelectKBest:
['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_9']

Random Forest Important Features:


,Feature,Importance
0,Feature_1,0.216891
5,Feature_6,0.188877
1,Feature_2,0.172183
8,Feature_9,0.095169
7,Feature_8,0.090471
3,Feature_4,0.077522
2,Feature_3,0.072877
9,Feature_10,0.033506
4,Feature_5,0.028018
6,Feature_7,0.024488


## Question 19 – Perform Binning and Frequency Encoding

1. Create age groups using binning.
2. Apply frequency encoding to ProductCategory.

In [19]:
encoding_df = pd.DataFrame({
    "Age": [18, 22, 27, 35, 42, 50, 61],
    "ProductCategory": [
        "Electronics", "Clothing", "Electronics",
        "Books", "Clothing", "Electronics", "Books"
    ]
})

encoding_df["AgeGroup"] = pd.cut(
    encoding_df["Age"],
    bins=[17, 25, 40, 60, float("inf")],
    labels=["Young", "Adult", "Middle-Aged", "Senior"]
)

frequency_mapping = encoding_df["ProductCategory"].value_counts()

encoding_df["CategoryFrequency"] = (
    encoding_df["ProductCategory"].map(frequency_mapping)
)

display(encoding_df)

,Age,ProductCategory,AgeGroup,CategoryFrequency
0,18,Electronics,Young,3
1,22,Clothing,Young,2
2,27,Electronics,Adult,3
3,35,Books,Adult,2
4,42,Clothing,Middle-Aged,2
5,50,Electronics,Middle-Aged,3
6,61,Books,Senior,2
